# Baseline Model: TF-IDF + Logistic Regression

This notebook trains and evaluates a baseline model using TF-IDF features and Logistic Regression.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.models.baseline import BaselineModel, train_baseline_model
from src.utils.metrics import calculate_metrics, print_metrics, confidence_analysis
from config.config import PROCESSED_DATA_DIR, MODELS_DIR

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Processed Data

In [ ]:
# Load processed data
train_df = pd.read_csv(PROCESSED_DATA_DIR / 'processed_train.csv')
val_df = pd.read_csv(PROCESSED_DATA_DIR / 'processed_val.csv')
test_df = pd.read_csv(PROCESSED_DATA_DIR / 'processed_test.csv')

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"\nSample data:")
train_df.head()

## 2. Train Baseline Model

In [ ]:
# Train model
model = train_baseline_model(
    train_df,
    val_df,
    test_df,
    text_column='text',
    label_column='label'
)

## 3. Model Evaluation

In [ ]:
# Get predictions
y_test = test_df['label']
y_pred = model.predict(test_df['text'].tolist())
y_proba = model.predict_proba(test_df['text'].tolist())

# Calculate metrics
metrics = model.evaluate(test_df['text'].tolist(), y_test)
print_metrics(metrics, "Baseline Model - Test Set")

## 4. Confusion Matrix

In [ ]:
# Plot confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test.map(model.label2id),
    [model.label2id[p] for p in y_pred]
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=model.classes,
    yticklabels=model.classes
)
plt.title('Confusion Matrix - Baseline Model', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Confidence Analysis

In [ ]:
# Analyze confidence
y_test_numeric = y_test.map(model.label2id).values
y_pred_numeric = np.array([model.label2id[p] for p in y_pred])

conf_analysis = confidence_analysis(y_proba, y_test_numeric, y_pred_numeric)
print("Confidence Analysis:")
print(conf_analysis)

# Plot
conf_analysis.plot(kind='bar', y='accuracy', figsize=(10, 6))
plt.title('Accuracy by Confidence Level', fontsize=14, fontweight='bold')
plt.xlabel('Confidence Range', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
# Get feature importance
feature_importance = model.get_feature_importance(top_n=10)

# Plot for each class
for class_name in model.classes[:3]:  # Show first 3 classes
    class_features = feature_importance[
        (feature_importance['class'] == class_name) &
        (feature_importance['type'] == 'positive')
    ].sort_values('coefficient', ascending=False).head(10)

    plt.figure(figsize=(10, 6))
    plt.barh(class_features['feature'], class_features['coefficient'])
    plt.title(f'Top Features for {class_name}', fontsize=14, fontweight='bold')
    plt.xlabel('Coefficient', fontsize=12)
    plt.tight_layout()
    plt.show()

## 7. Example Predictions

In [ ]:
# Test with custom examples
test_examples = [
    "I can't log into my account, please help!",
    "Where is my order? It's been 2 weeks!",
    "Your app keeps crashing on my phone",
    "I was charged twice for the same purchase"
]

predictions = model.predict(test_examples)
probabilities = model.predict_proba(test_examples)

print("Example Predictions:\n")
for text, pred, proba in zip(test_examples, predictions, probabilities):
    confidence = np.max(proba)
    print(f"Text: {text}")
    print(f"Prediction: {pred}")
    print(f"Confidence: {confidence:.4f}")
    print(f"All probabilities: {dict(zip(model.classes, proba))}")
    print("-" * 80)
    print()

## Summary

The baseline model provides:
- Fast training and inference
- Interpretable features
- Solid performance benchmark

Next: Train transformer model for improved accuracy